# Telangana Scholarship RAG Assistant (corrected & cleaned)

This is a rebuilt version of the original notebook. Three real bugs were fixed:

1. **Stale index** — metadata extraction (`scheme_id`) was being fixed mid-notebook but the
   Qdrant collection was never rebuilt from the fixed documents, so every indexed chunk had
   `scheme_id: None` forever.
2. **Silent scoring bug** — the final ranking function read a `"vector_score"` key that the
   retriever never produced (it produced `"score"`), so vector similarity contributed **0**
   to every ranking decision.
3. **Recall gap in pure dense retrieval** — short, terse rows (e.g. a single rejection-rule
   line) don't embed close to long natural-language questions. Dense search alone was
   silently dropping the correct chunk from the candidate list before reranking ever got a
   chance to see it. This version adds a **BM25 keyword search branch merged with dense
   vector search** at retrieval time, so exact-term matches (like "management quota") are
   never missed.

Structure is a single linear pipeline, top to bottom, no duplicate cells, no dead debug cells.


In [2]:
import re
import math
import time
from pathlib import Path
from collections import deque
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import fitz  # pymupdf
import tiktoken
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchAny
#from kaggle_secrets import UserSecretsClient
from groq import Groq

c:\Users\YASH PRABAKAR KATE\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup

In [3]:
from pathlib import Path

DATA_DIR = Path(r"D:\Downloads\RAG_PROJECT_TELANGANA_SS\data")
#PDF_PATH = Path(r"D:\Downloads\RAG_PROJECT_TELANGANA_SS\ilovepdf_merged.pdf")
QDRANT_PATH = "./qdrant_db"
COLLECTION_NAME = "scholarship_documents"

tokenizer = tiktoken.get_encoding("cl100k_base")


## 2. Load source documents

Both CSVs and the PDF are loaded now. The original notebook installed `pymupdf` but never
actually used it — the PDF's content was silently excluded from the knowledge base.

Critically, `load_csvs` now keeps the **raw row dict** in metadata (`raw_row`). That lets
`format_document` pull `scheme_id` / `scheme_name` / `category` directly from the original
CSV columns instead of regex-guessing them back out of rendered text later.

In [4]:
def load_csvs(data_dir):
    """Load every CSV into one text blob per row, keeping the raw row for later use."""
    documents = []
    csv_files = sorted(data_dir.rglob("*.csv"))
    print(f"Found {len(csv_files)} CSV files")

    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path)
            print(f"{csv_path.name}: {len(df)} rows")

            for index, row in df.iterrows():
                text_parts = []
                for column in df.columns:
                    value = row[column]
                    if pd.notna(value):
                        text_parts.append(f"{column}: {value}")

                text = " | ".join(text_parts)
                if not text.strip():
                    continue

                documents.append({
                    "text": text,
                    "metadata": {
                        "source": csv_path.name,
                        "row": index + 1,
                        "file_type": "csv",
                        # keep original columns so we never have to regex them back out
                        "raw_row": {
                            k: (None if pd.isna(v) else v)
                            for k, v in row.to_dict().items()
                        },
                    },
                })
        except Exception as e:
            print(f"Error reading {csv_path.name}: {e}")

    return documents


In [5]:
import fitz  # pymupdf

def load_pdf(pdf_path, min_chars=40):
    """Load a PDF page by page. Each page becomes one raw document."""
    if not pdf_path.exists():
        print(f"PDF not found at {pdf_path}, skipping.")
        return []

    documents = []
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if len(text) < min_chars:
                continue
            documents.append({
                "text": text,
                "metadata": {
                    "source": pdf_path.name,
                    "row": page_number,
                    "file_type": "pdf",
                    "raw_row": {},
                },
            })

    print(f"{pdf_path.name}: {len(documents)} pages with usable text")
    return documents


In [5]:
csv_documents = load_csvs(DATA_DIR)
#pdf_documents = load_pdf(PDF_PATH)

raw_documents = csv_documents
print("Total raw documents:", len(raw_documents))


Found 17 CSV files
application_process.csv: 8 rows
benefits.csv: 23 rows
citizen_charter_overview.csv: 6 rows
courses.csv: 5 rows
documents.csv: 43 rows
eligibility.csv: 24 rows
faqs_all_schemes.csv: 542 rows
hostel_diet_maintenance_rates.csv: 4 rows
official_contacts_grievances.csv: 2 rows
post_matric_hostels_2022_23.csv: 34 rows
pre_matric_hostels_2022_23.csv: 34 rows
procurement_policy.csv: 8 rows
rejection_rules.csv: 19 rows
rti_contacts.csv: 2 rows
schemes.csv: 24 rows
service_standards_timelines.csv: 7 rows
sources.csv: 13 rows
Total raw documents: 798


## 3. Clean text

In [6]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*\|\s*", " | ", text)
    return text.strip()


def clean_document(document):
    return {
        "text": clean_text(document["text"]),
        "metadata": document["metadata"],
    }


def remove_duplicates(documents):
    unique_documents = []
    seen = set()
    for document in documents:
        text = document["text"]
        if not text or text in seen:
            continue
        seen.add(text)
        unique_documents.append(document)
    return unique_documents


def clean_documents(documents):
    cleaned = [clean_document(d) for d in documents]
    cleaned = [d for d in cleaned if d["text"]]
    return remove_duplicates(cleaned)


cleaned_documents = clean_documents(raw_documents)
print("Documents before cleaning:", len(raw_documents))
print("Documents after cleaning:", len(cleaned_documents))


Documents before cleaning: 798
Documents after cleaning: 798


## 4. Format documents + extract metadata (fixed)

`scheme_id`, `scheme_name`, and `category` are now read straight from `raw_row` (the
original CSV columns) whenever available. Regex-on-rendered-text is kept only as a
fallback for PDF pages, which have no structured columns.

In [7]:
def clean_value(value):
    if value is None:
        return ""
    value = str(value).strip()
    return re.sub(r"\s+", " ", value)


def make_label(column_name):
    column_name = str(column_name).replace("_", " ").replace("-", " ")
    column_name = re.sub(r"\s+", " ", column_name)
    return column_name.strip().title()


def format_row_text(text):
    """'column: value | column: value' -> one 'Column: value' per line."""
    if not text:
        return ""
    fields = text.split("|")
    formatted_fields = []
    for field in fields:
        field = field.strip()
        if not field:
            continue
        if ":" in field:
            column, value = field.split(":", 1)
            value = clean_value(value)
            if value:
                formatted_fields.append(f"{make_label(column)}: {value}")
        else:
            formatted_fields.append(field)
    return "\n".join(formatted_fields)


def detect_document_type(source):
    source = source.lower()
    mapping = [
        ("faqs_all_schemes", "FAQ"),
        ("faq", "FAQ"),
        ("eligibility", "Eligibility"),
        ("benefit", "Benefits"),
        ("hostel", "Hostel Information"),
        ("maintenance", "Maintenance Rates"),
        ("course", "Courses"),
        ("document", "Required Documents"),
        ("application", "Application Process"),
        ("rejection", "Rejection Rules"),
        ("timeline", "Service Timeline"),
        ("service", "Service Timeline"),
        ("contact", "Contacts and Grievances"),
        ("grievance", "Contacts and Grievances"),
        ("rti", "RTI Information"),
        ("citizen", "Citizen Charter"),
        ("scheme", "Scholarship Scheme"),
        ("source", "Source Information"),
        ("ilovepdf", "Official PDF Document"),
    ]
    for key, doc_type in mapping:
        if key in source:
            return doc_type
    return "General Scholarship Information"


def extract_field(text, field_names):
    """Fallback regex extraction, used only when raw_row has nothing useful."""
    for field in field_names:
        pattern = rf"^{re.escape(field)}:\s*(.+)$"
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match:
            return match.group(1).strip()
    return None


def get_from_raw_row(raw_row, possible_columns):
    for name in possible_columns:
        if name in raw_row and raw_row[name] not in (None, ""):
            return str(raw_row[name]).strip()
    return None


def format_document(document):
    original_text = document.get("text", "")
    old_metadata = document.get("metadata", {})
    source = old_metadata.get("source", "unknown")
    row = old_metadata.get("row")
    raw_row = old_metadata.get("raw_row", {}) or {}
    file_type = old_metadata.get("file_type", "csv")

    formatted_text = format_row_text(original_text) if file_type == "csv" else original_text
    document_type = detect_document_type(source)

    scheme_id = get_from_raw_row(raw_row, ["scheme_id", "Scheme Id", "Scheme ID"]) \
        or extract_field(formatted_text, ["Scheme Id", "Scheme ID"])

    scheme_name = get_from_raw_row(raw_row, ["scheme_name", "Scheme Name", "scholarship_name"]) \
        or extract_field(formatted_text, ["Scheme Name", "Scholarship Name", "Scheme", "Scholarship Scheme"])

    category = get_from_raw_row(raw_row, ["category", "Category"]) \
        or extract_field(formatted_text, ["Category", "Categories"])

    metadata = {
        "source": source,
        "row": row,
        "file_type": file_type,
        "document_type": document_type,
        "scheme_name": scheme_name,
        "scheme_id": scheme_id,
        "category": category,
    }

    return {"text": formatted_text, "metadata": metadata}


def format_documents(documents):
    formatted = [format_document(d) for d in documents]
    return [d for d in formatted if d["text"].strip()]


def add_document_ids(documents):
    for i, document in enumerate(documents):
        document["metadata"]["document_id"] = f"doc_{i:06d}"
    return documents


formatted_documents = format_documents(cleaned_documents)
formatted_documents = add_document_ids(formatted_documents)
print("Formatted documents:", len(formatted_documents))

# Sanity check: scheme_id should now actually be populated
with_scheme_id = sum(1 for d in formatted_documents if d["metadata"]["scheme_id"])
print("Documents with a non-null scheme_id:", with_scheme_id)


Formatted documents: 798
Documents with a non-null scheme_id: 133


In [8]:
for doc in formatted_documents[:3]:
    print("\n" + "=" * 80)
    print(doc["metadata"])
    print(doc["text"][:400])



{'source': 'application_process.csv', 'row': 1, 'file_type': 'csv', 'document_type': 'Application Process', 'scheme_name': None, 'scheme_id': None, 'category': None, 'document_id': 'doc_000000'}
Step Number: 1
Process Stage: Online Registration
Responsible Party: Student
Action Description: Submit fresh/renewal form on ePASS portal; upload MeeSeva certificates and allotment orders
Statutory Timeline: Within 15 days of admission

{'source': 'application_process.csv', 'row': 2, 'file_type': 'csv', 'document_type': 'Application Process', 'scheme_name': None, 'scheme_id': None, 'category': None, 'document_id': 'doc_000001'}
Step Number: 2
Process Stage: Biometric Authentication
Responsible Party: Student / MeeSeva
Action Description: Perform biometric fingerprint authentication linked with Aadhaar
Statutory Timeline: At application submission

{'source': 'application_process.csv', 'row': 3, 'file_type': 'csv', 'document_type': 'Application Process', 'scheme_name': None, 'scheme_id': None,

## 5. Chunk documents

In [9]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(tokenizer.encode(text))


def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


def create_chunks(text, chunk_size=350, chunk_overlap=50):
    if not text:
        return []
    if count_tokens(text) <= chunk_size:
        return [text.strip()]

    sentences = split_into_sentences(text)
    chunks, current_sentences, current_tokens = [], [], 0

    for sentence in sentences:
        sentence_tokens = count_tokens(sentence)

        if sentence_tokens > chunk_size:
            if current_sentences:
                chunks.append(" ".join(current_sentences))
                current_sentences, current_tokens = [], 0
            words = sentence.split()
            temp = []
            for word in words:
                test = " ".join(temp + [word])
                if count_tokens(test) <= chunk_size:
                    temp.append(word)
                else:
                    if temp:
                        chunks.append(" ".join(temp))
                    temp = (temp[-10:] if temp else []) + [word]
            if temp:
                chunks.append(" ".join(temp))
            continue

        if current_tokens + sentence_tokens <= chunk_size:
            current_sentences.append(sentence)
            current_tokens += sentence_tokens
        else:
            if current_sentences:
                chunks.append(" ".join(current_sentences))
            overlap_sentences, overlap_tokens = [], 0
            for previous in reversed(current_sentences):
                previous_tokens = count_tokens(previous)
                if overlap_tokens + previous_tokens <= chunk_overlap:
                    overlap_sentences.insert(0, previous)
                    overlap_tokens += previous_tokens
                else:
                    break
            current_sentences = overlap_sentences + [sentence]
            current_tokens = overlap_tokens + sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return [c.strip() for c in chunks if c.strip()]


def chunk_documents(documents, chunk_size=350, chunk_overlap=50):
    chunked_documents = []
    for document in documents:
        text = document["text"]
        metadata = document["metadata"].copy()
        chunks = create_chunks(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

        for chunk_index, chunk in enumerate(chunks):
            chunk_metadata = metadata.copy()
            chunk_metadata["chunk_id"] = f"{metadata['document_id']}_chunk_{chunk_index}"
            chunk_metadata["chunk_index"] = chunk_index
            chunk_metadata["total_chunks"] = len(chunks)
            chunk_metadata["token_count"] = count_tokens(chunk)
            chunked_documents.append({"text": chunk, "metadata": chunk_metadata})

    return chunked_documents


chunked_documents = chunk_documents(formatted_documents, chunk_size=350, chunk_overlap=50)
print("Original documents:", len(formatted_documents))
print("Total chunks:", len(chunked_documents))


Original documents: 798
Total chunks: 798


## 6. Embeddings

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embedding_dimension = embedding_model.get_sentence_embedding_dimension()
print("Embedding dimension:", embedding_dimension)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 976.41it/s]


Embedding dimension: 384


C:\Users\YASH PRABAKAR KATE\AppData\Local\Temp\ipykernel_19072\804719362.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embedding_model.get_sentence_embedding_dimension()


In [11]:
texts = [doc["text"] for doc in chunked_documents]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

for document, embedding in zip(chunked_documents, embeddings):
    document["embedding"] = embedding.tolist()

print("Embedding shape:", embeddings.shape)


Batches: 100%|██████████| 25/25 [00:12<00:00,  2.04it/s]


Embedding shape: (798, 384)


## 7. Keyword (BM25) index

This is the fix for the recall gap. BM25 works on exact/lexical term overlap, so a short row
like `Rule Id: REJ04 | Rejection Reason: Management / Spot Admission Quota` gets found the
moment the query contains "management quota" — regardless of how far apart the two are in
embedding space.

In [12]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25_corpus_tokens = [tokenize(doc["text"]) for doc in chunked_documents]
bm25_index = BM25Okapi(bm25_corpus_tokens)
print("BM25 index built over", len(chunked_documents), "chunks")


BM25 index built over 798 chunks


## 8. Vector store (Qdrant) — fresh build with corrected metadata

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

QDRANT_PATH = "./qdrant_db"
COLLECTION_NAME = "scholarship_documents"
VECTOR_SIZE = embedding_dimension

client = QdrantClient(path=QDRANT_PATH)

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

points = []
for i, (document, embedding) in enumerate(zip(chunked_documents, embeddings)):
    payload = {"text": document["text"], **document["metadata"]}
    points.append(PointStruct(id=i, vector=embedding.tolist(), payload=payload))

BATCH_SIZE = 100
for start in range(0, len(points), BATCH_SIZE):
    batch = points[start:start + BATCH_SIZE]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)

print("Uploaded", len(points), "points")
print("scheme_id populated on upload:",
      sum(1 for p in points if p.payload.get("scheme_id")))


Uploaded 798 points
scheme_id populated on upload: 133


## 9. Hybrid retrieval

Dense vector search and BM25 keyword search each run independently, then get merged
(deduplicated by `chunk_id`) into a single candidate pool. This guarantees a document that's
an exact lexical match — even if it's a poor embedding match — still gets a chance to reach
the reranker.

In [16]:
def dense_search(query, top_k=30):
    query_embedding = embedding_model.encode(query, normalize_embeddings=True)
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
    ).points

    retrieved = []
    for result in results:
        payload = result.payload
        retrieved.append({
            "text": payload.get("text", ""),
            "vector_score": float(result.score),
            "metadata": {
                "source": payload.get("source"),
                "row": payload.get("row"),
                "document_type": payload.get("document_type"),
                "scheme_name": payload.get("scheme_name"),
                "scheme_id": payload.get("scheme_id"),
                "category": payload.get("category"),
                "document_id": payload.get("document_id"),
                "chunk_id": payload.get("chunk_id"),
            },
        })
    return retrieved


def keyword_search(query, top_k=30):
    scores = bm25_index.get_scores(tokenize(query))
    ranked_idx = np.argsort(scores)[::-1][:top_k]

    retrieved = []
    for idx in ranked_idx:
        if scores[idx] <= 0:
            continue
        document = chunked_documents[idx]
        retrieved.append({
            "text": document["text"],
            "bm25_score": float(scores[idx]),
            "metadata": document["metadata"],
        })
    return retrieved


from qdrant_client.models import Filter, FieldCondition, MatchAny

def scheme_filtered_search(query, scheme_ids, top_k=15):
    """Guaranteed-recall branch: always include the detected scheme's own top matches,
    regardless of how they rank against the rest of the corpus."""
    if not scheme_ids:
        return []

    query_embedding = embedding_model.encode(query, normalize_embeddings=True)
    scheme_filter = Filter(must=[FieldCondition(key="scheme_id", match=MatchAny(any=scheme_ids))])
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        query_filter=scheme_filter,
        limit=top_k,
    ).points

    retrieved = []
    for result in results:
        payload = result.payload
        retrieved.append({
            "text": payload.get("text", ""),
            "vector_score": float(result.score),
            "bm25_score": 0.0,
            "metadata": {
                "source": payload.get("source"),
                "row": payload.get("row"),
                "document_type": payload.get("document_type"),
                "scheme_name": payload.get("scheme_name"),
                "scheme_id": payload.get("scheme_id"),
                "category": payload.get("category"),
                "document_id": payload.get("document_id"),
                "chunk_id": payload.get("chunk_id"),
            },
        })
    return retrieved


def hybrid_retrieve(query, vector_k=30, bm25_k=30, scheme_k=15):
    dense_results = dense_search(query, top_k=vector_k)
    keyword_results = keyword_search(query, top_k=bm25_k)

    detected_schemes = detect_scheme(query)
    scheme_results = scheme_filtered_search(query, detected_schemes, top_k=scheme_k)

    merged = {}
    for r in dense_results:
        merged[r["metadata"]["chunk_id"]] = {**r, "bm25_score": 0.0}

    for r in keyword_results:
        chunk_id = r["metadata"]["chunk_id"]
        if chunk_id in merged:
            merged[chunk_id]["bm25_score"] = r["bm25_score"]
        else:
            merged[chunk_id] = {**r, "vector_score": 0.0}

    for r in scheme_results:
        chunk_id = r["metadata"]["chunk_id"]
        if chunk_id not in merged:
            merged[chunk_id] = r  # only add if not already found — don't downgrade a better score

    return list(merged.values())


In [17]:
# Quick check on the query that used to fail completely
query = "Are management quota students eligible for Post Matric Scholarship?"
candidates = hybrid_retrieve(query, vector_k=30, bm25_k=30)
print("Candidate pool size:", len(candidates))

for c in candidates:
    if "management" in c["text"].lower() or "spot admission" in c["text"].lower():
        print("\nFOUND:")
        print(c["text"])
        print(c["metadata"])
        print("vector_score:", c["vector_score"], "| bm25_score:", c["bm25_score"])


NameError: name 'detect_scheme' is not defined

## 10. Cross-encoder rerank

In [18]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, documents):
    if not documents:
        return []
    pairs = [[query, doc["text"]] for doc in documents]
    scores = reranker.predict(pairs)
    for document, score in zip(documents, scores):
        document["rerank_score"] = float(score)
    return documents


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1997.93it/s]


## 11. Query understanding (scheme / topic detection)

In [19]:
SCHEME_KEYWORDS = {
    "SCH01": ["post matric", "post-matric", "post matric scholarship", "pms",
              "college scholarship", "degree scholarship"],
    "SCH10": ["pre matric", "pre-matric", "pre matric scholarship"],
    "SCH05": ["overseas", "foreign university", "abroad", "overseas scholarship"],
}

TOPIC_KEYWORDS = {
    "income": ["income", "income limit", "family income", "annual income"],
    "attendance": ["attendance", "biometric", "75%"],
    "rejection": ["reject", "rejected", "rejection", "reason for rejection", "denied", "ineligible"],
    "quota": ["quota", "management quota", "category-b", "category b", "spot admission", "convener quota"],
    "documents": ["document", "certificate", "bonafide", "income certificate", "caste certificate"],
    "renewal": ["renewal", "renew", "previous year"],
    "bank": ["bank account", "passbook", "payment", "mtf"],
}


def detect_scheme(query):
    query_lower = query.lower()
    detected = []
    for scheme_id, keywords in SCHEME_KEYWORDS.items():
        if any(keyword in query_lower for keyword in keywords):
            detected.append(scheme_id)
    return detected


def detect_topics(query):
    query_lower = query.lower()
    detected = []
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword in query_lower for keyword in keywords):
            detected.append(topic)
    return detected


def classify_query(query):
    schemes = detect_scheme(query)
    topics = detect_topics(query)
    return {"schemes": schemes, "topics": topics, "scheme_specified": len(schemes) > 0}


## 12. Final ranking (single function, bug fixed)

The original had three overlapping ranking functions (`hybrid_rerank`, `final_rank`,
`scheme_aware_rank`) written at different points in the notebook, each slightly different,
with the last one silently reading a key (`vector_score`) that didn't exist in its input.
This is one function that combines everything and uses the correct key names throughout.

In [20]:
import math

def keyword_overlap_score(query, text):
    query_words = set(re.findall(r"\b[a-zA-Z0-9]+\b", query.lower()))
    text_words = set(re.findall(r"\b[a-zA-Z0-9]+\b", text.lower()))
    if not query_words:
        return 0.0
    return len(query_words & text_words) / len(query_words)


def scheme_match_score(query, metadata):
    detected_schemes = detect_scheme(query)
    if not detected_schemes:
        return 0.0
    return 1.0 if metadata.get("scheme_id") in detected_schemes else 0.0


def topic_match_score(query, text):
    topics = detect_topics(query)
    if not topics:
        return 0.0
    text_lower = text.lower()
    matches = 0
    for topic in topics:
        if any(keyword in text_lower for keyword in TOPIC_KEYWORDS[topic]):
            matches += 1
    return min(matches / len(topics), 1.0)


def final_rank(query, documents, top_k=5):
    ranked = []
    for document in documents:
        vector_score = document.get("vector_score", 0.0)
        bm25_score = document.get("bm25_score", 0.0)
        rerank_score = document.get("rerank_score", 0.0)

        # squash unbounded scores into roughly [0, 1]
        rerank_normalized = 1 / (1 + math.exp(-rerank_score))
        bm25_normalized = min(bm25_score / 10.0, 1.0)

        scheme_score = scheme_match_score(query, document["metadata"])
        topic_score = topic_match_score(query, document["text"])
        keyword_score = keyword_overlap_score(query, document["text"])

        final_score = (
            0.20 * vector_score
            + 0.15 * bm25_normalized
            + 0.25 * rerank_normalized
            + 0.15 * topic_score
            + 0.15 * scheme_score
            + 0.10 * keyword_score
        )

        document["scheme_score"] = scheme_score
        document["topic_score"] = topic_score
        document["keyword_score"] = keyword_score
        document["final_score"] = final_score
        ranked.append(document)

    ranked.sort(key=lambda x: x["final_score"], reverse=True)
    return ranked[:top_k]


In [21]:
# updated final_rank 
import math

def keyword_overlap_score(query, text):
    query_words = set(re.findall(r"\b[a-zA-Z0-9]+\b", query.lower()))
    text_words = set(re.findall(r"\b[a-zA-Z0-9]+\b", text.lower()))
    if not query_words:
        return 0.0
    return len(query_words & text_words) / len(query_words)


def scheme_match_score(query, metadata):
    detected_schemes = detect_scheme(query)
    if not detected_schemes:
        return 0.0
    return 1.0 if metadata.get("scheme_id") in detected_schemes else 0.0


def topic_match_score(query, text):
    topics = detect_topics(query)
    if not topics:
        return 0.0
    text_lower = text.lower()
    matches = 0
    for topic in topics:
        if any(keyword in text_lower for keyword in TOPIC_KEYWORDS[topic]):
            matches += 1
    return min(matches / len(topics), 1.0)
def final_rank(query, documents, top_k=5):
    ranked = []
    for document in documents:
        vector_score = document.get("vector_score", 0.0)
        bm25_score = document.get("bm25_score", 0.0)
        rerank_score = document.get("rerank_score", 0.0)

        rerank_normalized = 1 / (1 + math.exp(-rerank_score))
        bm25_normalized = min(bm25_score / 10.0, 1.0)

        scheme_score = scheme_match_score(query, document["metadata"])
        topic_score = topic_match_score(query, document["text"])
        keyword_score = keyword_overlap_score(query, document["text"])

        final_score = (
            0.20 * vector_score
            + 0.15 * bm25_normalized
            + 0.25 * rerank_normalized
            + 0.15 * topic_score
            + 0.15 * scheme_score
            + 0.10 * keyword_score
        )

        document["scheme_score"] = scheme_score
        document["topic_score"] = topic_score
        document["keyword_score"] = keyword_score
        document["final_score"] = final_score
        ranked.append(document)

    ranked.sort(key=lambda x: x["final_score"], reverse=True)

    # Guarantee: a chunk that exactly matches BOTH the detected scheme and the
    # detected topic is relevant almost by definition — don't let a noisy
    # cross-encoder score on oddly-formatted CSV text bury it.
    detected_schemes = detect_scheme(query)
    detected_topics = detect_topics(query)

    if detected_schemes and detected_topics:
        exact_match_ids = {
            d["metadata"]["chunk_id"] for d in ranked
            if d["scheme_score"] == 1.0 and d["topic_score"] == 1.0
        }
        if exact_match_ids:
            exact_matches = [d for d in ranked if d["metadata"]["chunk_id"] in exact_match_ids]
            others = [d for d in ranked if d["metadata"]["chunk_id"] not in exact_match_ids]
            selected = (exact_matches + others)[:top_k]
            selected.sort(key=lambda x: x["final_score"], reverse=True)
            return selected

    return ranked[:top_k]

In [84]:
# Same check as before, now through the full ranking pipeline
query = "Are management quota students eligible for Post Matric Scholarship?"
candidates = hybrid_retrieve(query, vector_k=30, bm25_k=30)
candidates = rerank_documents(query, candidates)
ranked = final_rank(query, candidates, top_k=5)

for i, r in enumerate(ranked):
    print("\n" + "=" * 80)
    print("RANK:", i + 1, "| final_score:", round(r["final_score"], 4))
    print("METADATA:", r["metadata"])
    print("TEXT:", r["text"])



RANK: 1 | final_score: 0.5624
METADATA: {'source': 'faqs_all_schemes.csv', 'row': 265, 'document_type': 'FAQ', 'scheme_name': None, 'scheme_id': None, 'category': 'Hyderabad Public Schools (HPS)', 'document_id': 'doc_000373', 'chunk_id': 'doc_000373_chunk_0'}
TEXT: Category: Hyderabad Public Schools (HPS)
Question: Are children admitted under the general quota eligible for scholarships?
Answer: Yes.

RANK: 2 | final_score: 0.5414
METADATA: {'source': 'faqs_all_schemes.csv', 'row': 3, 'document_type': 'FAQ', 'scheme_name': None, 'scheme_id': None, 'category': 'Post-Matric Scholarships Scheme', 'document_id': 'doc_000111', 'chunk_id': 'doc_000111_chunk_0'}
TEXT: Category: Post-Matric Scholarships Scheme
Question: Who is eligible?
Answer: Those pursuing post-secondary courses in recognized institutions/universities/colleges.

RANK: 3 | final_score: 0.4906
METADATA: {'source': 'rejection_rules.csv', 'row': 4, 'file_type': 'csv', 'document_type': 'Rejection Rules', 'scheme_name': None, 'sc

## 13. Build LLM context

In [22]:
def build_context(results):
    context_parts = []
    for i, result in enumerate(results):
        metadata = result["metadata"]
        header = f"""SOURCE {i + 1}
Source File: {metadata.get('source', 'Unknown')}
Document Type: {metadata.get('document_type', 'Unknown')}
"""
        if metadata.get("scheme_id"):
            header += f"Scheme ID: {metadata['scheme_id']}\n"
        if metadata.get("category"):
            header += f"Category: {metadata['category']}\n"

        context_parts.append(header + "\nContent:\n" + result["text"])

    return "\n\n" + "\n\n".join(context_parts)


## 14. LLM answer generation

Model name note: the original notebook used `gemini-3.6-flash` / `gemini-3.7-flash`, which
aren't published Gemini model IDs. Using `gemini-flash-latest`, an alias Google keeps
pointed at their current recommended Flash model, so this doesn't silently break when
specific version names get deprecated. Swap it for a pinned version if you need
reproducibility.

In [23]:
# do not run this cell 
load_dotenv()
groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Groq API key could not be loaded")
client_llm = Groq(api_key=groq_api_key)


In [24]:
load_dotenv()
groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Groq API key could not be loaded")
client_llm = Groq(api_key=groq_api_key)
print("Groq client initialized successfully")

Groq client initialized successfully


In [25]:
import time

def generate_answer(query, context, conversation_context="", max_retries=4):
    user_prompt = f"""{conversation_context}CURRENT USER QUESTION:
{query}

RETRIEVED CONTEXT (this is your ONLY source of facts — never the conversation above):
{context}

Answer the CURRENT question using only the retrieved context. Use the conversation
history above only to understand what "it", "that scheme", "what about ST students"
etc. refer to — never pull facts, numbers, or scheme rules from it.
"""
    for attempt in range(max_retries):
        try:
            response = client_llm.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.0,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 2 ** attempt
                print(f"Rate limited, retrying in {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Groq API unavailable after retries.")

In [29]:
# do not run this cell 
SYSTEM_PROMPT = """You are a Telangana Scholarship Information Assistant.

Your job is to answer questions ONLY using the retrieved official Telangana scholarship
context provided to you.

STRICT RULES:
1. Never invent information.
2. Never use outside knowledge.
3. Do not combine rules from different scholarship schemes unless the user explicitly
   asks for a comparison.
4. If a scheme is explicitly mentioned by the user, prioritize information belonging to
   that scheme.
5. If the user does not specify a scheme and the retrieved context contains different
   values for different schemes, clearly explain that the answer depends on the scheme.
6. Do not present a scheme-specific value as a universal value.
7. Preserve exact numbers, percentages, income limits and conditions from the retrieved
   context.
8. If the retrieved context is insufficient, say: "I could not find sufficient information
   in the available official sources."
9. Do not create citations, rules, scheme IDs or source files that are not present in the
   context.
10. Keep answers concise and easy to understand.
11. When useful, mention the source file the information came from.
12. If two sources contain different values, do NOT silently choose one. State that the
    sources differ and identify the relevant source/scheme for each value.
"""


def generate_answer(query, context):
    user_prompt = f"""USER QUESTION:
{query}

RETRIEVED CONTEXT:
{context}

Answer the user's question using only the retrieved context.
"""
    response = client_llm.models.generate_content(
        model="gemini-flash-latest",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
        ),
    )
    return response.text


In [36]:
%pip install pathlib 

Note: you may need to restart the kernel to use updated packages.


In [38]:
def generate_answer(query, context, conversation_context="", max_retries=4):
    user_prompt = f"""{conversation_context}CURRENT USER QUESTION:
{query}

RETRIEVED CONTEXT (this is your ONLY source of facts — never the conversation above):
{context}

Answer the CURRENT question using only the retrieved context. Use the conversation
history above only to understand what "it", "that scheme", "what about ST students"
etc. refer to — never pull facts, numbers, or scheme rules from it.
"""
    for attempt in range(max_retries):
        try:
            response = client_llm.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
                temperature=0.0,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                time.sleep(2 ** attempt)
            else:
                raise
    raise RuntimeError("Groq API unavailable after retries.")

## 15. Full pipeline

In [39]:
def ask_scholarship_assistant(query, vector_k=30, bm25_k=30, final_k=5):
    query_info = classify_query(query)

    candidates = hybrid_retrieve(query, vector_k=vector_k, bm25_k=bm25_k)
    candidates = rerank_documents(query, candidates)
    final_results = final_rank(query, candidates, top_k=final_k)

    context = build_context(final_results)
    answer = generate_answer(query, context)

    return {
        "query": query,
        "query_info": query_info,
        "answer": answer,
        "sources": final_results,
        "context": context,
    }


In [59]:
# do not run this cell
import requests

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {groq_api_key}"},
)
for m in response.json()["data"]:
    print(m["id"])

whisper-large-v3
whisper-large-v3-turbo
groq/compound-mini
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
qwen/qwen3.6-27b
allam-2-7b
groq/compound
openai/gpt-oss-20b
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m


## 16. Test it — including the query that used to fail

In [40]:
result = ask_scholarship_assistant("Who is eligible for Post Matric Scholarship?")
print(result["answer"])

**Eligibility for the Post‑Matric Scholarship (as per the official Telangana sources)**  

| Criterion | Detail |
|-----------|--------|
| **Course of study** | Must be pursuing post‑secondary (post‑matric) courses in a recognized institution, university or college. *(Source 1 – *faqs_all_schemes.csv*) |
| **Family income** | Annual household income must not exceed **Rs 2.00 Lakhs**. *(Source 2 – *eligibility.csv*) |
| **Age** | Must be of “college age” (the standard age range for higher‑education students). *(Source 2 – *eligibility.csv*) |
| **Academic status** | Must be admitted into a post‑matric course that is eligible for the Post‑Matric Scholarship. *(Source 2 – *eligibility.csv*) |
| **Caste/Community quota** | The scholarship is allocated according to the following community percentages: <br>‑ SC 75% <br>‑ SC‑Xian 2% <br>‑ BC 12% <br>‑ ST 6% <br>‑ Minorities 3% <br>‑ OC/Others 2% | *(Source 2 – *eligibility.csv*) |
| **Other condition** | The applicant must satisfy the above “

In [99]:
result = ask_scholarship_assistant("What is the income limit for SC students?")
print(result["answer"])

The income‑limit for SC students varies by scholarship scheme:

| Scheme (source) | Income limit for SC students |
|-----------------|------------------------------|
| **Financial Aid to SC Advocates** – *faqs_all_schemes.csv* (Source 1) | Rs. 2.00 lakhs per annum |
| **Telangana SC Study Circle** – *faqs_all_schemes.csv* (Source 2) | Below Rs. 3.00 lakhs per annum |
| **Kalyana Lakshmi Scheme** – *faqs_all_schemes.csv* (Source 3) | Rs. 2,00,000 (combined parental income) |
| **Corporate College Scheme** – *faqs_all_schemes.csv* (Source 4) | SC: below Rs. 2.00 lakhs per annum |
| **Pre‑Matric Scholarship** – *faqs_all_schemes.csv* (Source 5) | Rs. 2.00 lakhs per annum |

So, the applicable limit depends on which specific Telangana scholarship you are referring to.


In [63]:
result = ask_scholarship_assistant("Are management quota students eligible for Post Matric Scholarship?")
print(result["answer"])

No. Management‑quota (Category‑B) students are **ineligible** for the Post‑Matric Scholarship. The scheme’s rejection rules state that admission under “Management / Spot Admission Quota” leads to a rejection (Source 5, rejection_rules.csv).


In [90]:
result = ask_scholarship_assistant("What is the attendance requirement for Post Matric Scholarship?")
print(result["answer"])

The Post‑Matric Scholarship requires **a minimum of 75 % quarterly biometric attendance** for the student to remain eligible【Source 4: eligibility.csv (SCH01)】.  Falling below this 75 % threshold is listed as a reason for rejection【Source 5: rejection_rules.csv (SCH01)】.


In [97]:
result = ask_scholarship_assistant("What documents are required for Post Matric Scholarship?")
print(result["answer"])

**Required documents for the Post‑Matric Scholarship (Scheme SCH01)**  

| Document | Issuing Authority | Mandatory? | Purpose |
|----------|-------------------|------------|---------|
| Marks Memo (SSC / Intermediate / Degree) | State Board / University | **Yes** | Proof of academic qualification |
| Bonafide / Study Certificate | Concerned College / University | **Yes** | Confirmation of regular study and bonafide student status |
| Income Certificate | MeeSeva / Revenue Department | **Yes** | Verification of income‑ceiling eligibility |
| Caste Certificate | MeeSeva / Revenue Department | **Yes** | Verification of social status |
| Ration Card | Food & Civil Supplies Department | **No** (optional) | Entry of card number for social validation |

*Source: `documents.csv` (Scheme ID SCH01).*


In [96]:
result = ask_scholarship_assistant("Why can a scholarship application be rejected for Post‑Matric Scholarships")
print(result["answer"])

A Post‑Matric scholarship application can be rejected for the following reasons that are mentioned in the official FAQs:

1. **Rejected supporting documents** – If the Aadhaar, income‑certificate or caste certificate that you uploaded is not accepted, the application is marked as rejected. You must re‑upload the correct documents through the e‑PASS portal (Source 1: *faqs_all_schemes.csv* – Post‑Matric Scholarships Scheme).

2. **Verification failures** – After you submit the application, the college principal first verifies the documents against your application. If the college verification is not satisfactory, the application does not move forward. Then the Welfare Officer conducts a second scrutiny; if this officer does not confirm the college verification, the sanction (and thus the scholarship) will not be granted (Source 5: *faqs_all_schemes.csv* – Post‑Matric Scholarships Scheme).

So, a Post‑Matric scholarship can be rejected if any of the required certificates are rejected or 

In [94]:
result = ask_scholarship_assistant("What type of bank account is required?")
print(result["answer"])

The required bank‑account type differs by scheme:

| Scheme | Bank‑account requirement |
|--------|--------------------------|
| **Inter‑Caste Marriage Incentive Scheme** – a joint bank account is required【Source 1】. |
| **Pre‑Matric Scholarship** – an **Aadhaar‑linked** bank account (joint if the applicant is a minor)【Source 2】. |
| **Post‑Matric Scholarships Scheme** – the student must have an account in a **nationalised scheduled bank** to receive the scholarship【Source 3】. |
| **Financial Aid to SC Advocates** – a bank account is required, but the source does not specify a particular type【Source 4】. |
| **Corporate College Scheme** – the pocket‑expense amount is transferred directly to the **student’s bank account** (type not specified)【Source 5】. |

So, the exact type of account you need depends on which Telangana scholarship or aid you are applying for. If you tell me the specific scheme, I can give you the precise requirement.


In [93]:
result = ask_scholarship_assistant("Is a one year course eligible")
print(result["answer"])

No. According to the official data, courses that are **≤ 1 year** are placed in the “Non‑Eligible Programs” group (see *courses.csv*), and they trigger a rejection rule (REJ08) that marks any course of ≤ 1 year (or distance‑mode courses such as Open University or short‑term certificates) as **ineligible** under state guidelines (see *rejection_rules.csv*).


In [92]:
result = ask_scholarship_assistant("How do I apply for the scholarship?")
print(result["answer"])

The way you apply depends on which scholarship you are interested in.

**Pre‑Matric Scholarship**  
* Apply **only online** at **www.telanganaepass.cgg.gov.in**【Source 5】.  
* The online portal will guide you through entering your details and submitting the form.  
* (Registrations usually open in September/October each academic year【Source 2】.)  

**Post‑Matric Scholarships Scheme**  
* Apply through the same e‑PASS portal at **https://telanganaepass.cgg.gov.in**【Source 3】.  

So, choose the appropriate scheme and submit your application via the Telangana e‑PASS website.


In [91]:
# End of testing 
result = ask_scholarship_assistant("What benefits are provided?")
print(result["answer"])

The benefits that are mentioned differ from one scheme to another. Here’s what the official FAQs say for each scheme that appears in the retrieved documents:

| Scheme (source) | Benefits / facilities mentioned |
|-----------------|---------------------------------|
| **Telangana SC Study Circle** (faqs_all_schemes.csv) | Accommodation & food, classes, tests, book fund, Xerox material, personal allowance. |
| **Inter‑Caste Marriage Incentive Scheme** (faqs_all_schemes.csv) | Children of ICM couples are eligible for **all Government benefits**. |
| **Hostels – Admission, Facilities, Study, Menu, Health & Other Activities** (faqs_all_schemes.csv) | Basic infrastructure includes CCTV cameras, RO water plants, solar water heaters, gym material, bunker beds. |
| **Skill Up‑gradation Programme** (faqs_all_schemes.csv) | **No accommodation** is provided (the FAQ only states that accommodation is not offered). |
| **Financial Aid to SC Advocates** (faqs_all_schemes.csv) | **No accommodation**;

In [45]:
#query = "What is the attendance requirement for Post Matric Scholarship?"
query = "Are management quota students eligible for Post Matric Scholarship?"
candidates = hybrid_retrieve(query, vector_k=30, bm25_k=30)
candidates = rerank_documents(query, candidates)
results = final_rank(query, candidates, top_k=5)
#results = final_rank(query, top_k=15)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} | score: {doc.get('score')} ---")
    print("Scheme:", doc.get("scheme_id"))
    print(doc.get("text", doc.get("content", ""))[:1000])


--- Result 1 | score: None ---
Scheme: None
Category: Hyderabad Public Schools (HPS)
Question: Are children admitted under the general quota eligible for scholarships?
Answer: Yes.

--- Result 2 | score: None ---
Scheme: None
Category: Post-Matric Scholarships Scheme
Question: Who is eligible?
Answer: Those pursuing post-secondary courses in recognized institutions/universities/colleges.

--- Result 3 | score: None ---
Scheme: None
Category: Post-Matric Scholarships Scheme
Question: Is a Telangana student studying in another state eligible?
Answer: Yes, they can apply online.

--- Result 4 | score: None ---
Scheme: None
Category: Pre-Matric Scholarship
Question: Are private-school students eligible?
Answer: No — only Govt./local-body/Govt.-aided school students.

--- Result 5 | score: None ---
Scheme: None
Rule Id: REJ04
Scheme Id: SCH01
Rejection Reason: Management / Spot Admission Quota
Condition Trigger: Admitted under Category-B Management Quota or unofficial Spot Admission
Rectif

In [50]:
# 1. Does any chunk actually contain an attendance percentage?
for doc in chunked_documents:
    if "attend" in doc["text"].lower() and any(ch.isdigit() for ch in doc["text"]):
        print(doc["metadata"])
        print(doc["text"])
        print("---")

# 2. What scheme IDs actually exist in your data?
real_scheme_ids = {}
for doc in formatted_documents:
    sid = doc["metadata"].get("scheme_id")
    sname = doc["metadata"].get("scheme_name")
    if sid:
        real_scheme_ids.setdefault(sid, set()).add(sname)

for sid, names in real_scheme_ids.items():
    print(sid, "->", names)

{'source': 'application_process.csv', 'row': 3, 'file_type': 'csv', 'document_type': 'Application Process', 'scheme_name': None, 'scheme_id': None, 'category': None, 'document_id': 'doc_000002', 'chunk_id': 'doc_000002_chunk_0', 'chunk_index': 0, 'total_chunks': 1, 'token_count': 55}
Step Number: 3
Process Stage: College Verification
Responsible Party: College Principal / Nodal Officer
Action Description: Verify bonafide status, regular attendance (min 75%), certificates; upload hard copies with barcode
Statutory Timeline: Promptly upon registration
---
{'source': 'benefits.csv', 'row': 4, 'file_type': 'csv', 'document_type': 'Benefits', 'scheme_name': None, 'scheme_id': 'SCH05', 'category': None, 'document_id': 'doc_000011', 'chunk_id': 'doc_000011_chunk_0', 'chunk_index': 0, 'total_chunks': 1, 'token_count': 111}
Scheme Id: SCH05
Benefit Type: Exam Coaching Fee
Financial Amount: GRE: Rs. 9,000 (100 hrs); GMAT: Rs. 23,000 (100 hrs); TOEFL: Rs. 5,000 (60 hrs); IELTS: Rs. 4,200 (60 hrs)

In [80]:
query = "What is the attendance requirement for the Post-Matric Scholarship?"

# 1. Is a scheme even being detected?
print("Detected schemes:", detect_scheme(query))

# 2. Did you actually replace hybrid_retrieve with the scheme_filtered_search version
#    from a couple messages back? Check it's defined:
import inspect
print(inspect.signature(hybrid_retrieve))

# 3. Run retrieval + ranking directly, bypass the LLM entirely
candidates = hybrid_retrieve(query, vector_k=30, bm25_k=30)
print("Candidate pool size:", len(candidates))
candidates = rerank_documents(query, candidates)
results = final_rank(query, candidates, top_k=5)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} | final_score: {doc.get('final_score', 0):.4f} ---")
    print("Scheme:", doc["metadata"].get("scheme_id"), "| chunk_id:", doc["metadata"].get("chunk_id"))
    print(doc["text"][:300])

# 4. What actually gets sent to the LLM?
context = build_context(results)
print("\n\n=== FULL CONTEXT SENT TO LLM ===")
print(context)

Detected schemes: ['SCH01']
(query, vector_k=30, bm25_k=30, scheme_k=15)
Candidate pool size: 58

--- Result 1 | final_score: 0.6863 ---
Scheme: SCH15 | chunk_id: doc_000099_chunk_0
Scheme Id: SCH15
Income Ceiling Annual: Rs. 2.00 Lakhs
Age Limit: College age
Academic Criteria: Admitted into Post-Matric courses eligible for PMS
Caste Criteria: SC 75%, SC-Xian 2%, BC 12%, ST 6%, Minorities 3%, OC/Others 2%
Other Conditions: Must be eligible for Post-Matric Scholarship

--- Result 2 | final_score: 0.6379 ---
Scheme: None | chunk_id: doc_000340_chunk_0
Category: Pre-Matric Scholarship
Question: What are common reasons for rejection?
Answer: Receiving another scholarship; not a bonafide student; income above the limit; irregular attendance; discontinued study; incorrect details; missing hard copy.

--- Result 3 | final_score: 0.5704 ---
Scheme: None | chunk_id: doc_000110_chunk_0
Category: Post-Matric Scholarships Scheme
Question: What is the eligibility criteria?
Answer: Students of Telan

In [89]:
target_ids = {"doc_000085_chunk_0", "doc_000737_chunk_0"}

# Get the full scored+reranked list (not just top 5)
all_scored = final_rank(query, candidates, top_k=len(candidates))

for i, doc in enumerate(all_scored, 1):
    if doc["metadata"]["chunk_id"] in target_ids:
        print(f"Rank {i}/{len(all_scored)} | chunk_id: {doc['metadata']['chunk_id']}")
        print("  final_score:", round(doc['final_score'], 4))
        print("  vector_score:", round(doc.get('vector_score', 0), 4))
        print("  bm25_score:", round(doc.get('bm25_score', 0), 4))
        print("  rerank_score:", round(doc.get('rerank_score', 0), 4))
        print("  scheme_score:", round(doc.get('scheme_score', 0), 4))
        print("  topic_score:", round(doc.get('topic_score', 0), 4))
        print("  keyword_score:", round(doc.get('keyword_score', 0), 4))
        print()

# Also confirm: is "attendance" actually a detected topic for this query?
print("Detected topics:", detect_topics(query))

Rank 6/60 | chunk_id: doc_000085_chunk_0
  final_score: 0.4775
  vector_score: 0.7167
  bm25_score: 0.0
  rerank_score: -5.7055
  scheme_score: 1.0
  topic_score: 1.0
  keyword_score: 0.3333

Rank 23/60 | chunk_id: doc_000737_chunk_0
  final_score: 0.2793
  vector_score: 0.6465
  bm25_score: 0.0
  rerank_score: -11.0373
  scheme_score: 1.0
  topic_score: 0.0
  keyword_score: 0.0

Detected topics: ['quota']


In [100]:
import streamlit as st
from collections import deque

from rag_pipeline import (
    ask_scholarship_assistant,
    build_retrieval_query,
    build_conversation_context,
)

st.set_page_config(page_title="Telangana Scholarship Assistant", page_icon="🎓")
st.title("🎓 Telangana Scholarship Assistant")
st.caption("Ask about eligibility, income limits, attendance, documents, and rejection rules. Remembers your last 10 questions.")

MAX_TURNS = 10

# session_state persists across reruns within one browser session
if "memory" not in st.session_state:
    st.session_state.memory = deque(maxlen=MAX_TURNS)   # drives retrieval + reference resolution
if "display_messages" not in st.session_state:
    st.session_state.display_messages = []               # what's shown on screen

# Render past messages
for msg in st.session_state.display_messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("Ask a question...")

if user_input:
    # Show user message immediately
    with st.chat_message("user"):
        st.markdown(user_input)
    st.session_state.display_messages.append({"role": "user", "content": user_input})

    with st.chat_message("assistant"):
        with st.spinner("Looking it up..."):
            retrieval_query = build_retrieval_query(user_input, st.session_state.memory)
            conversation_context = build_conversation_context(st.session_state.memory)

            result = ask_scholarship_assistant(
                retrieval_query,
                conversation_context=conversation_context,
            )
            answer = result["answer"]

            st.markdown(answer)

            with st.expander("Sources used"):
                for src in result["sources"]:
                    meta = src["metadata"]
                    st.markdown(
                        f"**{meta.get('source')}** "
                        f"(scheme: {meta.get('scheme_id') or '—'}, "
                        f"type: {meta.get('document_type')})"
                    )
                    st.text(src["text"][:400])

    # Store in memory (deque auto-drops anything past 10)
    st.session_state.memory.append({"user": user_input, "answer": answer})
    st.session_state.display_messages.append({"role": "assistant", "content": answer})

# Sidebar controls
with st.sidebar:
    st.markdown(f"**Turns remembered:** {len(st.session_state.memory)}/{MAX_TURNS}")
    if st.button("Clear conversation"):
        st.session_state.memory.clear()
        st.session_state.display_messages = []
        st.rerun()

ModuleNotFoundError: No module named 'streamlit'